# Meuse heavy-metal points: mapping gradients before modeling

The classic Meuse dataset is a powerful teaching example for interpolation and environmental exposure mapping. This browser notebook uses a tiny local training table for the Meuse study area so learners can focus on map thinking before moving to full geostatistics.

The key design choice: map raw observations first; do not let an interpolated surface hide sparse sampling.

**Reflection questions:** Where are the high-zinc samples located relative to the river corridor? What sampling design would make interpolation more defensible? What uncertainty layer would you add?

In [ ]:
# Pyodide/JupyterLite bootstrap: install only pure-Python packages used in this notebook.
import sys, importlib
try:
    import micropip
except Exception:
    micropip = None

async def ensure_packages(packages):
    for pkg, import_name in packages:
        try:
            importlib.import_module(import_name)
        except Exception:
            if micropip is None:
                raise RuntimeError(f'{pkg} is not installed and micropip is unavailable.')
            await micropip.install(pkg)

await ensure_packages([('pandas','pandas'), ('folium','folium'), ('branca','branca'), ('plotly','plotly')])


In [ ]:
from pathlib import Path
import json, math, statistics
import pandas as pd
import folium
from folium.plugins import MarkerCluster, HeatMap, TimestampedGeoJson, MiniMap, Fullscreen, MeasureControl

DATA = Path('../data')

def load_json(name):
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def load_csv(name):
    return pd.read_csv(DATA / name)

def add_standard_controls(m):
    MiniMap(toggle_display=True).add_to(m)
    Fullscreen().add_to(m)
    MeasureControl(primary_length_unit='kilometers').add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m

def color_scale(values, colors=('green','orange','red')):
    vals = list(values)
    lo, hi = min(vals), max(vals)
    def pick(v):
        if hi == lo:
            return colors[1]
        t = (v - lo) / (hi - lo)
        return colors[0] if t < .33 else colors[1] if t < .66 else colors[2]
    return pick


In [ ]:
meuse = load_csv('meuse_training_points.csv')
scale = color_scale(meuse.zinc_mgkg)
m = folium.Map(location=[50.982,5.744], zoom_start=12, tiles='CartoDB positron')
# Approximate Meuse river centerline for context, not an administrative boundary.
river = [[50.955,5.700],[50.970,5.720],[50.985,5.740],[51.005,5.765]]
folium.PolyLine(river, weight=5, tooltip='Meuse river corridor context').add_to(m)
for _, r in meuse.iterrows():
    folium.CircleMarker([r.lat, r.lon], radius=5 + r.zinc_mgkg/180, fill=True, color=scale(r.zinc_mgkg), popup=f"{r.site}<br>Zinc {r.zinc_mgkg} mg/kg<br>{r.landuse}").add_to(m)
add_standard_controls(m)
m